## Importar librerías y cargar API key de Gemini

In [ ]:
!pip install openai

In [61]:
# Importación de librerías necesarias
from openai import OpenAI
from google.genai import types
from google.colab import userdata
import pandas as pd
import re
import tiktoken
from tqdm import tqdm
tqdm.pandas()

# Cargar API key de OpenAI desde variables seguras de Colab
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
# Inicializar cliente OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)
# Seleccionar modelo
# MODEL = "gpt-5.2"
MODEL = "gpt-5-mini"

## Cargar dataset

In [3]:
# conectar Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [74]:
# Cargar datasets individuales por tarea
path = "/content/drive/MyDrive/tfg/corpusMentalRiskES/splits/LLM/"
path_tara5 = path + "tara5/"

df_anx = pd.read_csv(path + "anxiety_llm.csv", sep=",")
df_dep = pd.read_csv(path + "depression_llm.csv", sep=",")
df_ed = pd.read_csv(path + "ed_llm.csv", sep=",")
df_multi = pd.read_csv(path + "multiclass_llm.csv", sep=",")

df_anx_tara5 = pd.read_csv(path_tara5 + "anxiety_llm_tara5.csv")
df_dep_tara5 = pd.read_csv(path_tara5 + "depression_llm_tara5.csv")
df_ed_tara5  = pd.read_csv(path_tara5 + "ed_llm_tara5.csv")
df_multi_tara5 = pd.read_csv(path_tara5 + "multiclass_llm_tara5.csv")

# Seleccionar tarea de clasificacion: anx | dep | ed | multi
task = "ed" # <--- Cambiar aquí

# Asignar el dataframe correspondiente a la tarea seleccionada
if task == "anx":
  df = df_anx
  df_tara5 = df_anx_tara5
elif task == "dep":
  df = df_dep
  df_tara5 = df_dep_tara5
elif task == "ed":
  df = df_ed
  df_tara5 = df_ed_tara5
elif task == "multi":
  df = df_multi
  df_tara5 = df_multi_tara5

## Función para construir el prompt

Se añade el texto a clasificar a un prompt fijo que cambio según la tarea.

In [75]:
def build_prompt(text, task):
  """ Construcción dinámica del prompt según tarea.
  Devuelve el texto completo que se enviará al modelo.
  """

  if task == "anx":
    return f"""
Vas a recibir un texto que corresponde a una serie de mensajes escritos por una persona en una red social.
El texto puede o no reflejar indicios de ansiedad.

La ansiedad se asocia a una preocupación persistente y difícil de controlar, acompañada de nerviosismo, inquietud interna y una sensación continua de amenaza o anticipación negativa. Estas experiencias suelen generar malestar emocional sostenido y dificultad para relajarse.

Tu tarea es clasificar el texto únicamente en una de las siguientes categorías:

1: el texto muestra indicios de ansiedad
0: el texto no muestra indicios de ansiedad

El contenido de los mensajes se proporciona a continuación entre comillas triples:

\"\"\"{text}\"\"\"

Responde solo con el número correspondiente.
"""
  elif task == "dep":
    return f"""
Vas a recibir un texto que corresponde a una serie de mensajes escritos por una persona en una red social.
El texto puede o no reflejar indicios de depresión.

La depresión se asocia a un estado persistente de tristeza, apatía o vacío emocional, acompañado de una disminución del interés o la capacidad de disfrute en actividades habituales. Estas experiencias pueden incluir pensamientos negativos recurrentes, fatiga emocional y una percepción pesimista de uno mismo o del futuro.

Tu tarea es clasificar el texto únicamente en una de las siguientes categorías:

1: el texto muestra indicios de depresión
0: el texto no muestra indicios de depresión

El contenido de los mensajes se proporciona a continuación entre comillas triples:

\"\"\"{text}\"\"\"

Responde solo con el número correspondiente.
"""
  elif task == "ed":
    return f"""
Vas a recibir un texto que corresponde a una serie de mensajes escritos por una persona en una red social.
El texto puede o no reflejar indicios de un trastorno de la conducta alimentaria.

Los trastornos de la conducta alimentaria se asocian a una preocupación persistente por el peso corporal, la alimentación o la imagen corporal, que puede manifestarse mediante pensamientos rígidos o negativos sobre la comida, el cuerpo o el control del peso. Estas preocupaciones suelen generar malestar emocional, culpa, ansiedad en torno a la alimentación y conductas de evitación o control.

Tu tarea es clasificar el texto únicamente en una de las siguientes categorías:

1: el texto muestra indicios de un trastorno de la conducta alimentaria
0: el texto no muestra indicios de un trastorno de la conducta alimentaria

El contenido de los mensajes se proporciona a continuación entre comillas triples:

\"\"\"{text}\"\"\"

Responde solo con el número correspondiente.
"""
  else:
    return f"""
Vas a recibir un texto que corresponde a una serie de mensajes escritos por una persona en una red social.
El texto puede reflejar indicios de uno de los siguientes estados psicológicos, o no reflejar ninguno de ellos.

Ansiedad: se asocia a una preocupación persistente y difícil de controlar, acompañada de nerviosismo, inquietud interna y una sensación continua de amenaza o anticipación negativa, generando malestar emocional y dificultad para relajarse.

Depresión: se asocia a un estado persistente de tristeza, apatía o vacío emocional, junto con una disminución del interés o la capacidad de disfrute en actividades habituales, pensamientos negativos recurrentes y una visión pesimista de uno mismo o del futuro.

Trastornos de la conducta alimentaria: se asocian a una preocupación persistente por el peso, la alimentación o la imagen corporal, que puede manifestarse mediante pensamientos rígidos o negativos sobre la comida o el cuerpo, así como malestar emocional, culpa o ansiedad en torno a la alimentación.

Tu tarea es clasificar el texto únicamente en una de las siguientes categorías:

0: el texto no muestra indicios de ninguno de los estados descritos
1: el texto muestra indicios de ansiedad
2: el texto muestra indicios de depresión
3: el texto muestra indicios de un trastorno de la conducta alimentaria

El contenido de los mensajes se proporciona a continuación entre comillas triples:

\"\"\"{text}\"\"\"

Responde solo con el número correspondiente.
"""

## Calcular número de tokens y coste

In [ ]:
# Obtener el tokenizador correspondiente al modelo
encoding = tiktoken.get_encoding("cl100k_base")

def total_tokens_dataset(df, task):
  """ Calcula el número total de tokens de entrada
  para un dataset completo, usando el tokenizador del modelo
  """
  total_tokens = 0

  for text in df["texto"]:
    prompt = build_prompt(text, task)
    tokens = len(encoding.encode(prompt))
    total_tokens += tokens

  return total_tokens


# Cálculo de tokens por dataset
total_anx = total_tokens_dataset(df_anx, "anx")
print("DATASET ANSIEDAD")
print("Tokens:", total_anx)
print("Media por texto:", total_anx / len(df_anx))

total_dep = total_tokens_dataset(df_dep, "dep")
print()
print("DATASET DEPRESIÓN")
print("Tokens:", total_dep)
print("Media por texto:", total_dep / len(df_dep))

total_ed = total_tokens_dataset(df_ed, "ed")
print()
print("DATASET ED")
print("Tokens:", total_ed)
print("Media por texto:", total_ed / len(df_ed))

total_multi = total_tokens_dataset(df_multi, "multi")
print()
print("DATASET MULTICLASE")
print("Tokens:", total_multi)
print("Media por texto:", total_multi / len(df_multi))

# Suma total de tokens necesarios para todo el experimento
print()
print(f"Total de tokens: {total_anx + total_dep + total_ed + total_multi}")

DATASET ANSIEDAD
Tokens: 511422
Media por texto: 1022.844

DATASET DEPRESIÓN
Tokens: 433676
Media por texto: 869.0901803607214

DATASET ED
Tokens: 299544
Media por texto: 894.1611940298508

DATASET MULTICLASE
Tokens: 1464529
Media por texto: 1097.8478260869565

Total de tokens: 2709171


## Clasificación zero-shot mediante modelos GPT

In [76]:
def classify_text(text, task):
    """Función de clasificación usando OpenAI.
    Devuelve:
    - int (0,1,2,3) si hay predicción válida
    - None si el modelo bloquea o no devuelve texto
    """
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "user", "content": build_prompt(text, task)}
            ],
            # GPT 5 mini solo acepta temperature=1
            temperature=0 if MODEL != "gpt-5-mini" else 1,
            # GPT 5 mini incluye tokens de razonamiento en max_completion_tokens,
            # por ello sube el nº de tokens a 500
            max_completion_tokens=5 if MODEL != "gpt-5-mini" else 500
        )

        output_text = response.choices[0].message.content

        if not output_text:
            print(f"No se ha devuelto texto: {response}")
            return None

        match = re.search(r"[0-3]", output_text)
        return int(match.group()) if match else None

    except Exception as e:
        print(f"Error al clasificar: {e}")
        return None


In [77]:
# Aplicar clasificación al dataset seleccionado
# Se almacena la predicción en una nueva columna "pred"
print(f"Iniciando predicción para el dataset: {task} | Total textos: {len(df)}")
df["pred"] = df["texto"].progress_apply(lambda x: classify_text(x, task))

Iniciando predicción para el dataset: ed | Total textos: 335


 31%|███▏      | 105/335 [03:11<10:58,  2.86s/it]

No se ha devuelto texto: ChatCompletion(id='chatcmpl-DsozWlSBY24gBhIM0h7UhXn9M7ww7', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1781958094, model='gpt-5-mini-2025-08-07', object='chat.completion', moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=500, prompt_tokens=660, total_tokens=1160, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=500, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


 35%|███▍      | 116/335 [03:34<09:20,  2.56s/it]

No se ha devuelto texto: ChatCompletion(id='chatcmpl-DsozthT2h6jL6neKfCPlsWsGFzHod', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1781958117, model='gpt-5-mini-2025-08-07', object='chat.completion', moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=500, prompt_tokens=434, total_tokens=934, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=500, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


...

 70%|██████▉   | 234/335 [07:38<04:47,  2.85s/it]

No se ha devuelto texto: ChatCompletion(id='chatcmpl-Dsp3qnJoFpWpqnWBCaeDnY8s3M0Ff', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1781958362, model='gpt-5-mini-2025-08-07', object='chat.completion', moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=500, prompt_tokens=632, total_tokens=1132, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=500, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


 75%|███████▍  | 251/335 [08:18<03:29,  2.49s/it]

No se ha devuelto texto: ChatCompletion(id='chatcmpl-Dsp4U8k8mRlTrE7mUqmCvUMwHQCmt', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1781958402, model='gpt-5-mini-2025-08-07', object='chat.completion', moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=500, prompt_tokens=565, total_tokens=1065, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=500, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


100%|██████████| 335/335 [10:44<00:00,  1.92s/it]


In [83]:
# Segundo intento para textos que fueron clasificados como NaN
mask = df["pred"].isna()

print(f"Textos pendientes: {mask.sum()}")

df.loc[mask, "pred"] = (
    df.loc[mask, "texto"]
      .progress_apply(lambda x: classify_text(x, task))
)

Textos pendientes: 1


100%|██████████| 1/1 [00:03<00:00,  3.52s/it]


### Comprobación de predicciones

In [84]:
from sklearn.metrics import f1_score, accuracy_score

# Reemplazar None por una clase inválida para que cuente como error
pred_labels = df["pred"].fillna(-1)

# Si es clasificación binaria
if task in ["anx", "dep", "ed"]:
    # Columna con la etiqueta real
    true_labels = df["bs"]
    f1 = f1_score(true_labels, pred_labels, average="macro")
else:
    # Columna con la etiqueta real
    true_labels = df["label_mc"]
    # Clasificación multiclase
    f1 = f1_score(true_labels, pred_labels, average="macro")

acc = accuracy_score(true_labels, pred_labels)

print("F1-score:", round(f1, 4))
print("Acc:", round(acc, 4))

F1-score: 0.8832
Acc: 0.8836


In [85]:
# Número de bloqueos
num_blocked = df["pred"].isna().sum()
block_rate = num_blocked / len(df)

print("Bloqueos:", num_blocked)
print("Tasa de bloqueo:", round(block_rate, 4))

Bloqueos: 0
Tasa de bloqueo: 0.0


### Almacenar predicciones

In [86]:
output_path = (
    f"/content/drive/MyDrive/tfg/corpusMentalRiskES/resultados/"
    f"{MODEL}_{task}.csv"
)
df.to_csv(output_path, index=False)

## Clasificación múltiple para Tara@5

Para analizar la estabilidad de las predicciones, se aplicará TARa@5 sobre un subconjunto estratificado del dataset. Cada ejemplo se evalúa cinco veces con el mismo modelo, reutilizando la primera predicción y generando cuatro ejecuciones adicionales, lo que permite medir el grado de consistencia de las salidas.

In [87]:
print(f"Iniciando TARa@5 para dataset: {task} | Textos: {len(df_tara5)}")

output_path = (
    f"/content/drive/MyDrive/tfg/corpusMentalRiskES/resultados/"
    f"{MODEL}_{task}_tara5.csv"
)

# Aplicar la misma función de clasificación 4 veces más
# Se almacena cada predicción en una nueva columna "pred_run_{num_run}"
for run in range(1, 5):
    col_name = f"pred_run{run+1}"

    if col_name in df_tara5.columns:
        print(f"{col_name} ya existe, se omite")
        continue

    print(f"\nEjecutando pred_run{run+1}...")

    df_tara5[col_name] = df_tara5["texto"].progress_apply(lambda x: classify_text(x, task))

    # Guardado incremental
    df_tara5.to_csv(output_path, index=False)

    print(f"Run {run+1} guardada correctamente")

Iniciando TARa@5 para dataset: ed | Textos: 100

Ejecutando pred_run2...


  8%|▊         | 8/100 [00:14<03:34,  2.34s/it]

No se ha devuelto texto: ChatCompletion(id='chatcmpl-Dsp983a5SXqjx0SH3SoJ1YApdOw4R', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1781958690, model='gpt-5-mini-2025-08-07', object='chat.completion', moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=500, prompt_tokens=937, total_tokens=1437, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=500, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


 23%|██▎       | 23/100 [00:45<03:15,  2.53s/it]

No se ha devuelto texto: ChatCompletion(id='chatcmpl-Dsp9dHSQLBRhZzb7tCvlCWIpjUhew', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1781958721, model='gpt-5-mini-2025-08-07', object='chat.completion', moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=500, prompt_tokens=792, total_tokens=1292, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=500, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


...

No se ha devuelto texto: ChatCompletion(id='chatcmpl-DspP0zKEq6eD6yUwZtyRNMc5GG9MR', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1781959674, model='gpt-5-mini-2025-08-07', object='chat.completion', moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=500, prompt_tokens=811, total_tokens=1311, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=500, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


 68%|██████▊   | 68/100 [03:19<01:54,  3.57s/it]

No se ha devuelto texto: ChatCompletion(id='chatcmpl-DspPlcXtXPv9R0cA3drCMOKkXBg4B', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1781959721, model='gpt-5-mini-2025-08-07', object='chat.completion', moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=500, prompt_tokens=632, total_tokens=1132, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=500, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


100%|██████████| 100/100 [04:57<00:00,  2.97s/it]

Run 5 guardada correctamente
